## 5. Exception Hierarchy

All Python exceptions form an **inheritance tree** rooted at `BaseException`. Understanding the hierarchy helps you write precise `except` clauses.

```
BaseException
├── SystemExit              ← raised by sys.exit()
├── KeyboardInterrupt       ← Ctrl+C
├── GeneratorExit           ← generator/coroutine closed
└── Exception               ← all "normal" exceptions
    ├── ArithmeticError
    │   ├── ZeroDivisionError
    │   ├── OverflowError
    │   └── FloatingPointError
    ├── LookupError
    │   ├── IndexError
    │   └── KeyError
    ├── OSError  (= IOError = EnvironmentError)
    │   ├── FileNotFoundError
    │   ├── PermissionError
    │   └── TimeoutError
    ├── ValueError
    ├── TypeError
    ├── NameError
    │   └── UnboundLocalError
    ├── AttributeError
    ├── ImportError
    │   └── ModuleNotFoundError
    ├── RuntimeError
    │   └── RecursionError
    └── StopIteration
```

### Key rules from the hierarchy
- Catching a **parent** class also catches all its **children**.
  - `except LookupError` catches both `IndexError` and `KeyError`.
  - `except OSError` catches `FileNotFoundError`, `PermissionError`, etc.
- **Never catch `BaseException`** — it swallows `KeyboardInterrupt` and `SystemExit`.
- Put **specific** exceptions before **broad** ones — Python matches the first one that fits.

In [1]:
# Parent catches children — LookupError catches IndexError & KeyError
def lookup_demo(container, key):
    try:
        return container[key]
    except LookupError as e:    # catches both IndexError and KeyError
        print(f"  LookupError ({type(e).__name__}): {e}")
        return None

lookup_demo([1, 2, 3], 10)          # IndexError
lookup_demo({"a": 1}, "b")          # KeyError

  LookupError (IndexError): list index out of range
  LookupError (KeyError): 'b'


In [2]:
# OSError parent catches file-related children
import errno

def open_file(path, mode="r"):
    try:
        return open(path, mode)
    except FileNotFoundError:
        print(f"  {path!r}: file does not exist")
    except PermissionError:
        print(f"  {path!r}: no permission")
    except OSError as e:       # catch-all for other OS errors
        print(f"  {path!r}: OS error [{e.errno}] {e.strerror}")
    return None

open_file("does_not_exist.txt")

  'does_not_exist.txt': file does not exist


In [3]:
# Wrong order — broad before specific (BAD)
print("Bad order (broad first):")
try:
    x = {}["key"]
except Exception as e:      # catches everything — KeyError never reached
    print(f"  Exception caught (too broad): {type(e).__name__}")
except KeyError:             # ← unreachable!
    print("  KeyError (never prints)")

Bad order (broad first):
  Exception caught (too broad): KeyError


In [4]:
# Correct order — specific first, broad last
print("Good order (specific first):")
try:
    x = {}["key"]
except KeyError as e:        # specific — matched first
    print(f"  KeyError caught: {e}")
except Exception as e:       # fallback for everything else
    print(f"  Unexpected error: {e}")

Good order (specific first):
  KeyError caught: 'key'


In [5]:
# Inspecting the MRO of an exception
print("FileNotFoundError MRO:")
for cls in FileNotFoundError.__mro__:
    print(f"  {cls.__name__}")


FileNotFoundError MRO:
  FileNotFoundError
  OSError
  Exception
  BaseException
  object


In [6]:
# isinstance check — useful for dynamic exception handling
errors = [ValueError("bad val"), KeyError("k"),
          IndexError(5), TypeError("wrong type")]

for err in errors:
    if isinstance(err, LookupError):
        print(f"  {type(err).__name__}: lookup problem — {err}")
    elif isinstance(err, (ValueError, TypeError)):
        print(f"  {type(err).__name__}: value/type problem — {err}")

  ValueError: value/type problem — bad val
  KeyError: lookup problem — 'k'
  IndexError: lookup problem — 5
  TypeError: value/type problem — wrong type
